# 18 — Diagnose KataGo losses and prepare stronger teacher data

## tl;dr

The completed benchmark lost all 60 KataGo games; 25 ended with zero AHR stones and 45 had a capture of at least five AHR stones after its first pass. These are development patterns, not proof that every pass caused a preventable loss. The new recipe uses searched teaching targets and a 90% strong-stratum mixture. The two-million-row bank is not prepared by this default Run All.

This diagnostic and data notebook preserves the frozen AHR player and prepares a new, immutable teaching bank. The completed notebook 17 matches are **consumed development evidence**, never training examples. The official dataset test payload remains unopened.

**Inputs:** pinned `quintic/go9x9` source release, notebooks 00–17 definition cells, frozen 8016/11724 checkpoints, and notebook 17's saved match report. **Outputs:** diagnostic report and figures, compatible train/dev mmap shards in `data/ahr_v2_teacher/prepared/<cache_id>`, producer checkpoints in `checkpoints/ahr-v2-k18-data-<cache_id>`, and validation receipts.

**Runtime:** diagnostic replay and short fixtures take minutes. Production preparation of two million positions can take hours and runs through notebook 20's detached controller. This notebook's Run All defaults to diagnostics and short validation; change the explicit `PREPARE_PRODUCTION` setting or use notebook 20 to prepare the main bank.

**Recovery:** rerun `go2_k18_prepare(PROJECT, K18_CONFIG)` with the identical config and source definitions. It resumes the latest hash-verified producer checkpoint; a mismatch raises an error. One producer owns each cache. Complete game/100-second commit boundaries preserve RNG, source cursors, accepted counts, immutable shard hashes and diagnostics. The latest two checkpoints and the best are retained.

In [ ]:
from pathlib import Path
import os, nbformat, json
PROJECT=Path.cwd()
while not (PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb').exists():
    if PROJECT.parent==PROJECT:raise FileNotFoundError('Open this notebook inside the AHR Go workspace.')
    PROJECT=PROJECT.parent
_boot={'__name__':'ahr_k18_bootstrap','PROJECT':PROJECT}
for _cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',as_version=4).cells:
    if _cell.cell_type=='code' and 'ahr-export' in _cell.metadata.get('tags',[]):exec(compile(_cell.source,'notebook00-export','exec'),_boot)
for _name in ['01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_counterfactual_search.ipynb','05_evaluation_and_progress.ipynb','08_repaired_tactics_and_search.ipynb','12_teacher_dataset_and_targets.ipynb','13_teacher_pretraining_and_recovery.ipynb','14_worker_reading_and_search.ipynb','15_teacher_evaluation_and_launch.ipynb','16_play_teacher_trained_player.ipynb','17_strength_and_speed_benchmarks.ipynb']:
    if not (PROJECT/'experiments/ahr_v2'/ _name).exists():
        # Actual historical notebook names are resolved from their immutable prefix.
        _matches=list((PROJECT/'experiments/ahr_v2').glob(_name[:2]+'_*.ipynb'))
        if len(_matches)!=1:raise FileNotFoundError(_name)
        _name=_matches[0].name
    _boot['go2_import_exports'](PROJECT/'experiments/ahr_v2'/_name,_boot)
globals().update({k:v for k,v in _boot.items() if k!='__name__'})
EXECUTION_MODE=os.environ.get('AHR_K18_MODE','diagnose')
PREPARE_PRODUCTION=False


## Context and methods

The new mixture assigns **90% of accepted rows to opponent strata 4–7**, compared with 50% previously. Within training games it favors competitive teacher estimates and retains late-game cleanup coverage. On valid teacher turns, searched policy/value replace the raw estimate; all other turns retain a labeled raw fallback. Legal masks and causal histories are reconstructed under our no-suicide positional-superko rules.

Teacher values remain estimates from the publisher's native rules. Policy support is renormalized on our legal actions; observed outcome and ownership supervision is enabled only when the complete legal replay's terminal score agrees with the publisher. A stronger analysis query uses our explicit rules, but KataGo still uses its own pass-alive evaluation; it does not prove that a group survives or establish a terminal result.

Train and development opening families remain disjoint. Test records may appear in downloaded indices but their game payloads are not opened. Benchmark 17's pass/capture diagnosis cannot establish that every pass caused the later loss.

The strongest stratum contains fewer games than other strata. A metadata inventory caps it below 800,000 usable rows at 96 positions per game; a 500-game whole-replay check accepted 499 games. Its production quota is 600,000 rows, leaving a reserve rather than requiring an impossible 800,000-row quota.

In [ ]:
import copy, math, statistics, collections, selectors, subprocess, uuid, time, shutil, gzip, hashlib, io
from collections import OrderedDict
GO2_K18_NOTEBOOK='18_diagnosis_and_stronger_teacher_data.ipynb'
GO2_K18_DEFAULT={
    'cache_id':'go9x9-competitive-v2','train_positions':2000000,'dev_positions':16384,
    'max_positions_per_game':96,'games_per_commit':128,'checkpoint_seconds':100.0,
    'seed':2026101118,'opponent_weights':[.025,.025,.025,.025,.10,.20,.30,.30],
    'competitive_abs_value':.90,'competitive_selection_boost':4.0,
    'cleanup_fraction':.65,'cleanup_selection_boost':2.0,
    'searched_selection_boost':1.5,'competitive_loss_weight':1.5,
    'cleanup_loss_weight':1.15,'searched_loss_weight':1.1,
    'policy':'valid_searched_teacher_turn_else_raw_renormalized',
    'value':'valid_searched_teacher_turn_else_raw_current_player',
    'purpose':'development','test_consumed':False}

def go2_k18_namespace(project):
    project=Path(project);namespace={'__name__':'ahr_k18_exports','PROJECT':project}
    for cell in nbformat.read(project/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',as_version=4).cells:
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'notebook00-export','exec'),namespace)
    for prefix in ['01','02','03','05','08','12','13','14','15','16','17','18']:
        paths=list((project/'experiments/ahr_v2').glob(prefix+'_*.ipynb'))
        if len(paths)!=1:raise RuntimeError('Ambiguous notebook prefix '+prefix)
        namespace['go2_import_exports'](paths[0],namespace)
    return namespace

def go2_k18_sources(project):
    root=Path(project)/'experiments/ahr_v2';names=[]
    for prefix in ['00','01','02','03','05','08','12','13','14','15','16','17','18']:
        matches=list(root.glob(prefix+'_*.ipynb'))
        if len(matches)!=1:raise RuntimeError('Ambiguous source notebook '+prefix)
        names.append(matches[0].name)
    return {name:go2_code_sha(root/name) for name in names}

def go2_k18_config(config=None):
    result=copy.deepcopy(GO2_K18_DEFAULT);result.update(config or {})
    unknown=set(result)-set(GO2_K18_DEFAULT)
    if unknown:raise ValueError('Unknown data recipe fields: '+str(sorted(unknown)))
    go2_teacher_cache_path(Path('.'),result['cache_id'])
    for field in ['train_positions','dev_positions','max_positions_per_game','games_per_commit']:
        if type(result[field]) is not int or result[field]<1:raise ValueError('Invalid '+field)
    weights=np.asarray(result['opponent_weights'],dtype=float)
    if weights.shape!=(8,) or not np.isfinite(weights).all() or (weights<=0).any():raise ValueError('Eight positive opponent weights required.')
    result['opponent_weights']=(weights/weights.sum()).tolist()
    if not 0<result['competitive_abs_value']<=1 or not 0<=result['cleanup_fraction']<=1:raise ValueError('Invalid competitive/cleanup thresholds.')
    if not 1<=result['checkpoint_seconds']<=120:raise ValueError('Producer checkpoint interval must be at most 120 seconds.')
    for field in ['competitive_selection_boost','cleanup_selection_boost','searched_selection_boost','competitive_loss_weight','cleanup_loss_weight','searched_loss_weight']:
        if not math.isfinite(result[field]) or not 0<result[field]<=10:raise ValueError('Invalid weight '+field)
    if result['purpose']!='development' or result['test_consumed'] is not False:raise ValueError('Only development preparation; test remains unopened.')
    return result

GO2_K18_FIELDS=dict(GO2_TEACHER_FIELDS)
GO2_K18_FIELDS.update({'policy_searched':(np.bool_,()),'row_weight':(np.float32,()),
    'competitive':(np.bool_,()),'cleanup':(np.bool_,()),'source_teacher_visits':(np.int16,())})

def go2_k18_quota(total,weights):
    exact=np.asarray(weights,dtype=float)*int(total);counts=np.floor(exact).astype(int)
    for i in np.argsort(-(exact-counts),kind='stable')[:int(total)-int(counts.sum())]:counts[i]+=1
    return counts.tolist()

def go2_k18_convert(source_root,row,config,split='train'):
    if row['split']=='test':raise ValueError('Test payload cannot be converted.')
    metadata,arrays=go2_teacher_read_game(Path(source_root),row)
    length=go2_teacher_validate_arrays(metadata,arrays)
    searched=np.asarray(arrays['search_policy_valid'],dtype=bool)
    sv=np.asarray(arrays['search_value'][searched],dtype=float)
    sp=np.asarray(arrays['search_policy'][searched],dtype=float)
    if not np.isfinite(sv).all() or (np.abs(sv)>1+1e-6).any():raise ValueError('Invalid searched value.')
    if len(sp) and (not np.isfinite(sp).all() or (sp<0).any() or not np.allclose(sp.sum(1),1,atol=1e-5,rtol=0)):
        raise ValueError('Invalid searched policy distribution.')
    states=[];state=go2_initial_state(9,7.5)
    try:
        for t,action in enumerate(arrays['actions']):
            board=tuple(-1 if v==2 else int(v) for v in arrays['stones'][t].reshape(-1))
            if state.terminal or state.board!=board:return {'accepted':False,'reason':'board_or_terminal_history_incompatible','game_id':row['game_id'],'ply':t}
            states.append(state);state=go2_step(state,int(action))
    except Go2IllegalMove as exc:
        return {'accepted':False,'reason':'illegal_under_our_rules','game_id':row['game_id'],'ply':len(states)-1,'detail':str(exc)}
    if not state.terminal:return {'accepted':False,'reason':'not_terminal_under_our_rules','game_id':row['game_id']}
    values=np.where(searched,arrays['search_value'],arrays['raw_value'])
    competitive=np.abs(values)<=config['competitive_abs_value']
    cleanup=np.arange(length)>=math.floor(length*config['cleanup_fraction'])
    take=min(config['max_positions_per_game'],length)
    if split=='train':
        weights=np.ones(length,dtype=float)
        weights*=np.where(competitive,config['competitive_selection_boost'],1.)
        weights*=np.where(cleanup,config['cleanup_selection_boost'],1.)
        weights*=np.where(searched,config['searched_selection_boost'],1.)
        seed=int(hashlib.sha256((str(config['seed'])+':'+row['game_id']).encode()).hexdigest()[:16],16)
        rng=np.random.default_rng(seed)
        selected=np.sort(rng.choice(length,take,replace=False,p=weights/weights.sum()))
    else:selected=np.unique(np.linspace(0,length-1,take,dtype=np.int64))
    final=go2_score(state);winner=1 if final['score_black']>0 else -1 if final['score_black']<0 else 0
    observed_valid=bool(np.isfinite(metadata['white_score']) and abs(float(metadata['white_score'])+final['score_black'])<=1e-5)
    result={k:np.empty((len(selected),)+shape,dtype=dtype) for k,(dtype,shape) in GO2_K18_FIELDS.items()}
    for i,t in enumerate(selected):
        pos=states[int(t)];features=go2_encode_state(pos);legal=np.asarray(go2_legal_mask(pos),dtype=bool)
        policy=(arrays['search_policy'][t] if searched[t] else arrays['raw_policy'][t]).astype(np.float64)*legal
        mass=float(policy.sum())
        if mass<=1e-12:return {'accepted':False,'reason':'no_compatible_teacher_policy_mass','game_id':row['game_id'],'ply':int(t)}
        policy/=mass
        if not np.array_equal(features[16].astype(bool).reshape(81),legal[:81]):raise AssertionError('History/features/legal mismatch.')
        score=float(arrays['raw_score'][t]);score_valid=bool(arrays['raw_score_valid'][t] and math.isfinite(score) and abs(score)<=88.5)
        observed=pos.to_play*winner
        result['features'][i]=features;result['legal'][i]=legal;result['policy'][i]=policy
        result['value'][i]=values[t];result['score'][i]=score/88.5 if score_valid else 0
        result['score_valid'][i]=score_valid;result['outcome'][i]=[observed==-1,observed==0,observed==1]
        result['outcome_valid'][i]=observed_valid
        result['ownership'][i]=np.asarray(final['ownership_black'],dtype=np.int8).reshape(9,9)*pos.to_play
        result['ownership_valid'][i]=observed_valid;result['game_id'][i]=row['game_id'].encode()
        result['family'][i]=row['opening_family'].encode();result['ply'][i]=t;result['opponent'][i]=row['opponent_index']
        result['teacher_turn'][i]=(1 if pos.to_play==1 else 2)==metadata['expert_color']
        result['removed_policy_mass'][i]=max(0.,1.-mass);result['policy_searched'][i]=searched[t]
        result['competitive'][i]=competitive[t];result['cleanup'][i]=cleanup[t]
        result['source_teacher_visits'][i]=min(int(arrays['teacher_visits'][t]),32767)
        weight=(config['competitive_loss_weight'] if competitive[t] else 1.)*(config['cleanup_loss_weight'] if cleanup[t] else 1.)*(config['searched_loss_weight'] if searched[t] else 1.)
        result['row_weight'][i]=weight if split=='train' else 1.
    return {'accepted':True,'arrays':result,'rows':len(selected),'game_id':row['game_id'],
        'family':row['opening_family'],'source_split':row['split'],'opponent_index':int(row['opponent_index']),
        'source_game_rows':length,'observed_auxiliary_valid':observed_valid,
        'our_terminal_score_black':final['score_black'],'publisher_terminal_score_white':metadata['white_score']}


## Commit an immutable bank and exact source cursor

The new cache preserves the existing bank. Float16 feature planes and memory-mapped shards keep the production bank near eight GB rather than loading it into RAM. A producer checkpoint is committed only after every referenced file is flushed and hashed. Uncommitted shard directories are removed only when resuming the same producer under its lock.

In [ ]:
def go2_k18_verify_shards(root,shards):
    root=Path(root)
    for shard in shards:
        for field,artifact in shard['files'].items():
            path=root/artifact['path']
            if not path.is_file() or path.stat().st_size!=artifact['bytes'] or go2_file_sha(path)!=artifact['sha256']:
                raise ValueError('K18 shard checksum mismatch: '+str(path))
            array=np.load(path,mmap_mode='r',allow_pickle=False);dtype,shape=GO2_K18_FIELDS[field]
            if array.shape!=(shard['rows'],)+shape or array.dtype!=np.dtype(dtype):raise ValueError('K18 shard schema mismatch: '+field)
        artifact=shard['games_artifact'];path=root/artifact['path']
        if not path.is_file() or go2_file_sha(path)!=artifact['sha256']:raise ValueError('K18 game provenance checksum mismatch.')
    return True

def go2_k18_prepare(project,config=None,max_games=None,on_commit=None):
    project=Path(project);config=go2_k18_config(config)
    root=go2_teacher_cache_path(project,config['cache_id']);root.mkdir(parents=True,exist_ok=True)
    checkpoint=project/'checkpoints'/('ahr-v2-k18-data-'+config['cache_id'])
    identity={'recipe':config,'sources':go2_k18_sources(project),'teacher':GO2_TEACHER_HASH,'release':GO2_TEACHER_RELEASE,'schema':'ahr_k18_producer_v1'}
    if max_games is not None and (type(max_games) is not int or max_games<1):raise ValueError('max_games must be positive or None.')
    if (root/'manifest.json').exists():
        dataset=go2_k18_dataset(project,config['cache_id'],verify=True)
        if dataset['manifest']['preparation_identity']!=go2_digest(identity):raise ValueError('Completed cache identity/config mismatch.')
        return dataset
    bytes_per_row=sum(np.dtype(dtype).itemsize*int(np.prod(shape) if shape else 1) for dtype,shape in GO2_K18_FIELDS.values())
    # Estimate only remaining writes; old immutable banks are never removed.
    if shutil.disk_usage(root).free<(config['train_positions']+config['dev_positions'])*bytes_per_row+2*1024**3:
        raise RuntimeError('Insufficient persistent storage for this bank plus safety reserve.')
    source,records=go2_teacher_index(project)
    ordered={s:go2_teacher_balanced_order(records,'train' if s=='train' else 'validation',config['seed']) for s in ['train','dev']}
    targets={'train':go2_k18_quota(config['train_positions'],config['opponent_weights']),
             'dev':go2_k18_quota(config['dev_positions'],config['opponent_weights'])}
    with go2_checkpoint_lock(checkpoint):
        saved,entry=go2_load_checkpoint(checkpoint,identity=identity,restore_rng=True)
        if saved is None:
            cursor={'source_cursors':{'train':0,'dev':0},'counts':{'train':[0]*8,'dev':[0]*8},
                'accepted_games':{'train':0,'dev':0},'families':{'train':[],'dev':[]},'rejected':{},'shards':[],
                'searched_rows':{'train':0,'dev':0},'competitive_rows':{'train':0,'dev':0},
                'cleanup_rows':{'train':0,'dev':0},'auxiliary_rows':{'train':0,'dev':0},
                'native_value_rows':{'train':0,'dev':0},'commits':0,'attempts':0,'status':'PREPARING',
                'selection_rng':'independent PCG64 seeded by SHA256(seed,game_id); no cross-game RNG position',
                'inflight_game':None}
        else:
            cursor=copy.deepcopy(saved['sampler'])
            if saved['config']!=config:raise RuntimeError('Producer config changed; restart refused.')
            go2_k18_verify_shards(root,cursor['shards'])
        referenced={s['directory'] for s in cursor['shards']}
        for path in root.glob('shard-*'):
            if path.name not in referenced:
                if not path.is_dir():raise RuntimeError('Uncommitted shard is not a directory.')
                shutil.rmtree(path)
        current_attempts=0;started=time.monotonic()
        while any(cursor['counts'][s]!=targets[s] for s in ['train','dev']):
            split=next(s for s in ['train','dev'] if cursor['counts'][s]!=targets[s])
            batches=[];provenance=[];chunk_start=time.monotonic();chunk_attempts=0
            while chunk_attempts<config['games_per_commit'] and time.monotonic()-chunk_start<config['checkpoint_seconds']:
                if cursor['counts'][split]==targets[split] or (max_games is not None and current_attempts>=max_games):break
                position=cursor['source_cursors'][split]
                if position>=len(ordered[split]):raise RuntimeError('Not enough compatible games to fill immutable strong-stratum quotas.')
                row=ordered[split][position];cursor['source_cursors'][split]+=1
                opponent=int(row['opponent_index']);remaining=targets[split][opponent]-cursor['counts'][split][opponent]
                if remaining<=0:continue
                chunk_attempts+=1;current_attempts+=1;cursor['attempts']+=1
                result=go2_k18_convert(source['root'],row,config,split)
                if not result['accepted']:
                    reason=result['reason'];cursor['rejected'][reason]=cursor['rejected'].get(reason,0)+1;continue
                take=min(remaining,result['rows']);batch={k:v[:take] for k,v in result['arrays'].items()}
                batches.append(batch);provenance.append({k:v for k,v in result.items() if k!='arrays'}|{'committed_rows':take})
                cursor['counts'][split][opponent]+=take;cursor['accepted_games'][split]+=1;cursor['families'][split].append(result['family'])
                for total,field in [('searched_rows','policy_searched'),('competitive_rows','competitive'),('cleanup_rows','cleanup'),('auxiliary_rows','outcome_valid')]:
                    cursor[total][split]+=int(batch[field].sum())
                cursor['native_value_rows'][split]+=take
            cursor['commits']+=1
            if batches:
                directory=f"shard-{cursor['commits']:08d}-{split}";partial=root/(directory+'.partial');partial.mkdir()
                files={};rows_count=sum(len(b['value']) for b in batches)
                for field in GO2_K18_FIELDS:
                    path=partial/(field+'.npy');array=np.concatenate([b[field] for b in batches])
                    with path.open('wb') as handle:np.save(handle,array,allow_pickle=False);handle.flush();os.fsync(handle.fileno())
                    files[field]={'path':directory+'/'+path.name,'sha256':go2_file_sha(path),'bytes':path.stat().st_size}
                go2_atomic_json(partial/'games.json',provenance);games=partial/'games.json'
                artifact={'path':directory+'/games.json','sha256':go2_file_sha(games),'bytes':games.stat().st_size}
                os.replace(partial,root/directory)
                cursor['shards'].append({'split':split,'directory':directory,'rows':rows_count,'files':files,'games_artifact':artifact})
            if set(cursor['families']['train'])&set(cursor['families']['dev']):raise AssertionError('Opening-family train/development leak.')
            complete=all(cursor['counts'][s]==targets[s] for s in ['train','dev'])
            cursor['status']='COMPLETE' if complete else 'PREPARING';cursor['updated_at_utc']=go2_now()
            metrics={'counts':copy.deepcopy(cursor['counts']),'attempts':cursor['attempts'],'rejected':copy.deepcopy(cursor['rejected']),
                'searched_rows':copy.deepcopy(cursor['searched_rows']),'competitive_rows':copy.deepcopy(cursor['competitive_rows']),
                'cleanup_rows':copy.deepcopy(cursor['cleanup_rows']),'auxiliary_rows':copy.deepcopy(cursor['auxiliary_rows']),
                'elapsed_this_invocation_seconds':time.monotonic()-started,'status':cursor['status']}
            state={'schema':GO2_SCHEMA,'identity':identity,'model':{'external_teacher_sha256':GO2_TEACHER_HASH},
                'optimizer':{},'scheduler':{},'scaler':None,'rng':go2_capture_rng(),
                'sampler':copy.deepcopy(cursor),'global_step':cursor['commits'],'config':copy.deepcopy(config),'metrics':metrics,
                'replay':{'source_release':GO2_TEACHER_RELEASE,'complete_game_boundary':True}}
            go2_save_checkpoint(checkpoint,state,best=complete,resume_cell=GO2_K18_NOTEBOOK+': go2_k18_prepare(PROJECT, identical saved config)')
            go2_atomic_json(root/'preparation_status.json',metrics|{'checkpoint_directory':str(checkpoint),'preparation_identity':go2_digest(identity)})
            if on_commit:on_commit(metrics)
            if max_games is not None and current_attempts>=max_games and not complete:
                return {'status':'PREPARING','cache_path':str(root),'preparation_identity':go2_digest(identity),'cursor':copy.deepcopy(cursor)}
        go2_k18_verify_shards(root,cursor['shards'])
        manifest={'schema':'ahr_k18_teacher_cache_v1','status':'COMPLETE','config':config,'sources':identity['sources'],
            'preparation_identity':go2_digest(identity),'source_release_id':GO2_TEACHER_RELEASE,
            'teacher_sha256':GO2_TEACHER_HASH,'source_inventory_sha256':go2_digest(source['inventory']['files']),
            'counts':{s:sum(cursor['counts'][s]) for s in ['train','dev']},'opponent_counts':cursor['counts'],
            'families':{s:sorted(set(cursor['families'][s])) for s in ['train','dev']},
            'accepted_games':cursor['accepted_games'],'rejected':cursor['rejected'],'shards':cursor['shards'],
            'target_counts':{k:cursor[k] for k in ['searched_rows','competitive_rows','cleanup_rows','auxiliary_rows','native_value_rows']},
            'target_provenance':{'policy':'searched on valid teacher turns, labeled raw fallback otherwise; exact legal renormalization',
                'value':'native-rule teacher estimate, searched on valid teacher turns, raw otherwise; current-player signed',
                'score':'available raw native-rule teacher estimate, normalized by88.5 and masked',
                'outcome':'terminal replay verified only on publisher-score agreement',
                'ownership':'terminal replay verified only on publisher-score agreement',
                'row_weight':'bounded emphasis on competitive, cleanup and searched rows; dev weights1'},
            'benchmark17_training_rows':0,'test_consumed':False,'created_utc':go2_now()}
        manifest['identity']=go2_digest(manifest);go2_atomic_json(root/'manifest.json',manifest)
    return go2_k18_dataset(project,config['cache_id'],verify=True)

def go2_k18_dataset(project,cache_id='go9x9-competitive-v2',verify=True):
    root=go2_teacher_cache_path(project,cache_id)
    if not (root/'manifest.json').exists():raise FileNotFoundError('K18 bank is not complete: '+str(root))
    manifest=json.loads((root/'manifest.json').read_text());signed=dict(manifest);identity=signed.pop('identity',None)
    if identity!=go2_digest(signed):raise ValueError('K18 manifest signature mismatch.')
    if manifest['schema']!='ahr_k18_teacher_cache_v1' or manifest['status']!='COMPLETE' or manifest['test_consumed'] is not False:
        raise ValueError('K18 dataset schema/status/test contract mismatch.')
    if manifest['sources']!=go2_k18_sources(project):raise ValueError('K18 data/source mismatch; silent feature-code replacement refused.')
    if set(manifest['families']['train'])&set(manifest['families']['dev']):raise ValueError('K18 opening-family leakage.')
    if verify:go2_k18_verify_shards(root,manifest['shards'])
    dataset={'manifest':manifest,'identity':identity,'cache_path':str(root),'status':'COMPLETE','_mmap_cache':OrderedDict()}
    for split in ['train','dev']:
        shards=[s for s in manifest['shards'] if s['split']==split];ends=np.cumsum([s['rows'] for s in shards],dtype=np.int64)
        count=int(ends[-1]) if len(ends) else 0
        if count!=manifest['counts'][split]:raise ValueError('K18 split count mismatch.')
        dataset[split]={'count':count,'row_count':count,'shards':shards,'ends':ends,
            'family_count':len(manifest['families'][split]),'game_count':manifest['accepted_games'][split]}
    return dataset

def go2_k18_sample_batch(dataset,indices,device='cpu',split='train'):
    if split not in ['train','dev']:raise ValueError('Only training and development batches; test remains unopened.')
    if isinstance(indices,torch.Tensor):indices=indices.detach().cpu().numpy()
    indices=np.asarray(indices)
    if indices.ndim!=1 or indices.dtype.kind not in 'iu' or not len(indices):raise ValueError('Nonempty integer indices required.')
    view=dataset[split]
    if (indices<0).any() or (indices>=view['count']).any():raise IndexError('K18 batch outside split.')
    fields=['features','legal','policy','value','score','score_valid','outcome','outcome_valid','ownership','ownership_valid',
        'row_weight','policy_searched','teacher_turn','opponent','ply','removed_policy_mass','competitive','cleanup']
    buffers={k:np.empty((len(indices),)+GO2_K18_FIELDS[k][1],dtype=GO2_K18_FIELDS[k][0]) for k in fields}
    shard_ids=np.searchsorted(view['ends'],indices,side='right');cache=dataset['_mmap_cache'];root=Path(dataset['cache_path'])
    for shard_id in np.unique(shard_ids):
        key=(split,int(shard_id));shard=view['shards'][int(shard_id)]
        if key not in cache:cache[key]={k:np.load(root/shard['files'][k]['path'],mmap_mode='r',allow_pickle=False) for k in fields}
        mapped=cache[key];cache.move_to_end(key);positions=np.flatnonzero(shard_ids==shard_id)
        start=0 if shard_id==0 else int(view['ends'][shard_id-1]);local=indices[positions]-start
        for k in fields:buffers[k][positions]=mapped[k][local]
        while len(cache)>16:cache.popitem(last=False)
    booleans={'legal','score_valid','outcome_valid','ownership_valid','policy_searched','teacher_turn','competitive','cleanup'}
    integers={'opponent','ply'}
    batch={k:torch.as_tensor(v,device=device,dtype=torch.bool if k in booleans else torch.int64 if k in integers else torch.float32) for k,v in buffers.items()}
    go2_teacher_validate_batch(batch)
    if not torch.isfinite(batch['row_weight']).all() or not (batch['row_weight']>0).all():raise ValueError('Invalid K18 row weights.')
    if (batch['policy_searched']&~batch['teacher_turn']).any():raise ValueError('Searched target used on an invalid teacher turn.')
    return batch


## Stronger teacher queries under explicit AHR rules

The optional API sends complete move history, not a fabricated move order. Its searched distributions and regret estimates are teaching signals, not game proofs. Queries use a deterministic pinned single-thread CPU backend, a bounded timeout and traceable request/response hashes. A timeout produces an exception and no completed teaching row. The engine is closed by a context manager.

In [ ]:
class Go2K18Teacher:
    def __init__(self,project,visits=64,timeout_seconds=120,seed=2026101118,assets=None,env=None,gpu_device=0):
        self.project=Path(project);self.assets=copy.deepcopy(assets) if assets is not None else go2_bench_katago_assets(project,allow_download=False)
        if self.assets.get('model_sha256')!=GO2_TEACHER_HASH or go2_file_sha(self.assets['model'])!=GO2_TEACHER_HASH:raise RuntimeError('Teacher model identity mismatch.')
        if self.assets.get('executable_sha256') and go2_file_sha(self.assets['executable'])!=self.assets['executable_sha256']:raise RuntimeError('Teacher executable checksum mismatch.')
        for field in ['binary_pins','runtime_sha256']:
            for asset_path,sha in self.assets.get(field,{}).items():
                if Path(asset_path).is_absolute() and go2_file_sha(asset_path)!=sha:raise RuntimeError('Teacher dependency checksum mismatch: '+asset_path)
        self.environment=dict(os.environ);self.environment.update(env or {})
        self.gpu_device=int(gpu_device)
        self.visits=int(visits);self.timeout_seconds=float(timeout_seconds);self.sequence=0;self.buffer=bytearray()
        if self.visits<2 or self.timeout_seconds<=0:raise ValueError('At least two teacher visits and a positive timeout required.')
        directory=self.project/'reports/ahr_v2/k18_teacher_configs';directory.mkdir(parents=True,exist_ok=True)
        settings={'numAnalysisThreads':1,'numSearchThreadsPerAnalysisThread':1,'numEigenThreadsPerModel':1,
            'nnMaxBatchSize':1,'nnCacheSizePowerOfTwo':12,'nnMutexPoolSizePowerOfTwo':8,
            'maxVisits':self.visits,'nnRandomize':False,'nnRandSeed':str(seed),'searchRandSeed':str(seed),
            'rootNumSymmetriesToSample':1,'rootSymmetryPruning':False,'rootNoiseEnabled':False,
            'chosenMoveTemperature':0,'chosenMoveTemperatureEarly':0,'reportAnalysisWinratesAs':'SIDETOMOVE',
            'logToStderr':False,'logAllRequests':False,'logAllResponses':False,'logSearchInfo':False,
            'startupPrintMessageToStderr':False,'wideRootNoise':0,'ignorePreRootHistory':False,
            'conservativePass':False,'preventCleanupPhase':True,'useEvalCache':False}
        if 'CUDA' in self.assets.get('backend',''):
            settings.update({'cudaDeviceToUse':self.gpu_device,'cudaUseFP16':False,'cudaUseNHWC':False})
        text='\n'.join(k+' = '+(str(v).lower() if isinstance(v,bool) else str(v)) for k,v in settings.items())+'\n'
        self.config_path=directory/('analysis-'+hashlib.sha256(text.encode()).hexdigest()[:16]+'.cfg')
        if self.config_path.exists() and self.config_path.read_text()!=text:raise RuntimeError('Teacher config collision.')
        if not self.config_path.exists():go2_atomic_bytes(self.config_path,text.encode())
        self.identity={'assets':self.assets,'rules':copy.deepcopy(GO2_BENCH_KATA_RULES),'config_sha256':go2_file_sha(self.config_path),'visits':self.visits}
        self.stderr_path=directory/('analysis-'+str(os.getpid())+'-'+str(time.monotonic_ns())+'.log');self.stderr=self.stderr_path.open('xb')
        self.process=subprocess.Popen([self.assets['executable'],'analysis','-model',self.assets['model'],'-config',str(self.config_path),'-quit-without-waiting'],
            cwd=directory,stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=self.stderr,bufsize=0,env=self.environment)
        self.selector=selectors.DefaultSelector();self.selector.register(self.process.stdout,selectors.EVENT_READ)
    def query(self,state,visits=None,allowed_actions=None):
        if state.size!=9 or state.terminal:raise ValueError('Teacher requires a live 9x9 state.')
        self.sequence+=1;visits=int(visits or self.visits)
        if visits<2:raise ValueError('Teacher analysis needs at least two visits.')
        player=state.initial_to_play;moves=[]
        for action in state.actions:
            moves.append(['B' if player==1 else 'W',_go2_bench_gtp_vertex(int(action),9)]);player=-player
        initial=[['B' if v==1 else 'W',_go2_bench_gtp_vertex(i,9)] for i,v in enumerate(state.initial_board) if v]
        request={'id':str(self.sequence),'moves':moves,'initialStones':initial,'initialPlayer':'B' if state.initial_to_play==1 else 'W',
            'rules':copy.deepcopy(GO2_BENCH_KATA_RULES),'komi':state.komi,'boardXSize':9,'boardYSize':9,
            'analyzeTurns':[len(moves)],'maxVisits':visits,'includeOwnership':True,'includePolicy':True,'analysisPVLen':8}
        legal=np.asarray(go2_legal_mask(state),dtype=bool)
        if allowed_actions is not None:
            allowed=[int(a) for a in allowed_actions]
            if not allowed or any(a<0 or a>81 or not legal[a] for a in allowed):raise ValueError('Forced analysis includes an illegal action.')
            request['allowMoves']=[{'player':'B' if state.to_play==1 else 'W','moves':[_go2_bench_gtp_vertex(a,9) for a in allowed],'untilDepth':1}]
        deadline=time.monotonic()+self.timeout_seconds;start=time.monotonic()
        try:self.process.stdin.write((json.dumps(request,separators=(',',':'))+'\n').encode());self.process.stdin.flush()
        except (BrokenPipeError,OSError) as exc:raise RuntimeError('Teacher analysis exited.') from exc
        warnings=[]
        while True:
            while b'\n' not in self.buffer:
                remaining=deadline-time.monotonic()
                if remaining<=0 or not self.selector.select(max(0,remaining)):
                    self.close();raise TimeoutError('Teacher analysis UNKNOWN on timeout.')
                chunk=os.read(self.process.stdout.fileno(),65536)
                if not chunk:raise RuntimeError('Teacher analysis closed output: '+self.stderr_path.read_text()[-1000:])
                self.buffer.extend(chunk)
                if len(self.buffer)>16*1024*1024:raise RuntimeError('Teacher response exceeds bound.')
            line,_,tail=self.buffer.partition(b'\n');self.buffer=bytearray(tail)
            response=json.loads(line)
            if response.get('id') not in [None,request['id']]:raise RuntimeError('Teacher response ID mismatch.')
            if 'error' in response:raise RuntimeError('Teacher analysis error: '+response['error'])
            if 'warning' in response:
                warnings.append(response)
                raise RuntimeError('Teacher warning may change rules/config; strict analysis refused: '+str(response))
            if response.get('isDuringSearch',False):continue
            if 'rootInfo' in response:break
        root=response['rootInfo']
        if root.get('currentPlayer')!=('B' if state.to_play==1 else 'W'):raise RuntimeError('Teacher perspective mismatch.')
        policy=np.zeros(82,dtype=np.float64);move_values={};move_scores={}
        for item in response.get('moveInfos',[]):
            action=_go2_bench_gtp_action(item['move'],9)
            if legal[action]:
                policy[action]+=max(0,float(item.get('edgeVisits',item['visits'])))
                move_values[action]=2*float(item['winrate'])-1;move_scores[action]=float(item['scoreLead'])
        if policy.sum()<=0:raise RuntimeError('Teacher completed without any compatible searched action.')
        policy/=policy.sum();value=2*float(root['winrate'])-1;score=float(root['scoreLead'])
        ownership=np.asarray(response.get('ownership',[]),dtype=np.float32)
        if not math.isfinite(value) or abs(value)>1.000001 or not math.isfinite(score):raise RuntimeError('Teacher emitted nonfinite value/score.')
        if ownership.shape!=(81,) or not np.isfinite(ownership).all() or (np.abs(ownership)>1.00001).any():raise RuntimeError('Teacher ownership missing/invalid.')
        result={'policy':policy.astype(np.float32),'policy_valid':True,'value':float(value),'score':float(np.clip(score/88.5,-1,1)),
            'score_points':score,'score_valid':abs(score)<=88.5,'ownership':ownership.reshape(9,9),
            'ownership_valid':True,'teacher_estimate':True,'move_values':move_values,'move_scores':move_scores,
            'visits':int(root['visits']),'best_action':int(np.argmax(policy)),'query':request,
            'response':response,'query_sha256':go2_digest(request),'response_sha256':go2_digest(response),
            'identity':self.identity,'elapsed_seconds':time.monotonic()-start,
            'provenance':'KataGo searched estimate under explicit AHR rules, native pass-alive evaluation; no terminal proof'}
        return result
    def regret(self,state,chosen_action,visits=None):
        best=self.query(state,visits=visits);forced=self.query(state,visits=visits,allowed_actions=[int(chosen_action)])
        return {'best':best,'chosen':forced,'estimated_winrate_regret':max(0.,(best['value']-forced['value'])/2),
            'estimated_score_regret':best['score_points']-forced['score_points'],'certainty':'teacher_estimate'}
    def close(self):
        process=getattr(self,'process',None)
        if process is not None:
            if process.poll() is None:
                process.terminate()
                try:process.wait(timeout=2)
                except subprocess.TimeoutExpired:process.kill();process.wait(timeout=2)
            for stream in [process.stdin,process.stdout]:
                if stream:stream.close()
        if hasattr(self,'selector'):self.selector.close()
        if hasattr(self,'stderr'):self.stderr.close()
    def __enter__(self):return self
    def __exit__(self,*args):self.close()


## Results: replay the consumed matches and select diagnostic positions

Positions are selected before AHR's first pass and before a capture of at least five AHR stones. They remain development-only. Optional Scout/search comparison records disagreement and a teacher regret estimate on the same causal history. The saved match report did not record the original neural outputs, so re-evaluated diagnostics are identified explicitly.

In [ ]:
def go2_k18_diagnosis(project,report_path=None):
    project=Path(project);path=Path(report_path or project/'reports/ahr_v2/benchmarks/ahr-v2-benchmark-20261009-v1/report.json')
    report=json.loads(path.read_text());details=[];positions=[]
    for game in report['games']:
        state=go2_initial_state(9,7.5);color=game['ahr_color'];first_pass=None;big_captures=[];pass_count=0;lost=0;captured=0
        for ply,item in enumerate(game['trace'],1):
            if state.to_play!=item['color']:raise AssertionError('Trace player mismatch.')
            before=state.board.count(color);opp_before=state.board.count(-color)
            if item['action']==81 and item.get('ahr_move',False):
                pass_count+=1
                if first_pass is None:
                    first_pass={'ply':ply,'stones':before,'score':go2_score(state)['score_black']*color}
                    if 'katago' in game['profile']:positions.append({'game_index':game['game_index'],'profile':game['profile'],'reason':'before_first_ahr_pass',
                        'state':go2_state_to_dict(state),'chosen_action':81,'purpose':'consumed_development_only'})
            prior=state;state=go2_step(state,item['action'])
            n=before-state.board.count(color)+(item['color']==color and item['action']!=81)
            lost+=max(0,n);captured+=max(0,opp_before-state.board.count(-color)+(item['color']==-color and item['action']!=81))
            if n>=5:
                big_captures.append({'ply':ply,'stones':n,'after_first_pass':first_pass is not None})
                if 'katago' in game['profile'] and len(big_captures)==1:
                    # The pre-capture opponent turn is useful for group evaluation;
                    # its preceding AHR turn is reconstructed independently below.
                    actions=list(prior.actions);pre=go2_initial_state(9,7.5)
                    for action in actions[:-1]:pre=go2_step(pre,action)
                    if pre.to_play==color:positions.append({'game_index':game['game_index'],'profile':game['profile'],'reason':'ahr_turn_before_first_large_capture',
                        'state':go2_state_to_dict(pre),'chosen_action':int(actions[-1]),'purpose':'consumed_development_only'})
        if list(state.board)!=game['state']['board'] or go2_score(state)['score_black']*color!=game['score_for_ahr']:raise AssertionError('Final board/score mismatch.')
        details.append({'profile':game['profile'],'game_index':game['game_index'],'ahr_color':color,'outcome':game['outcome'],
            'score':game['score_for_ahr'],'final_ahr_stones':state.board.count(color),'ahr_passes':pass_count,
            'first_pass_ply':first_pass['ply'] if first_pass else None,'first_pass_score':first_pass['score'] if first_pass else None,
            'plies_after_first_pass':game['plies']-first_pass['ply'] if first_pass else None,
            'large_capture':bool(big_captures),'large_capture_after_pass':any(c['after_first_pass'] for c in big_captures),
            'ahr_stones_captured':lost,'opponent_stones_captured':captured,'terminal_two_passes':state.consecutive_passes==2})
    summary=[]
    for profile in sorted({r['profile'] for r in details}):
        rows=[r for r in details if r['profile']==profile]
        summary.append({'Opponent':profile,'Games':len(rows),'AHR wins':sum(r['outcome']=='WIN' for r in rows),
            'Median score margin':statistics.median(r['score'] for r in rows),'Zero AHR stones':sum(r['final_ahr_stones']==0 for r in rows),
            'Large capture after first pass':sum(r['large_capture_after_pass'] for r in rows),
            'Median AHR passes':statistics.median(r['ahr_passes'] for r in rows),
            'Median plies after first pass':statistics.median(r['plies_after_first_pass'] for r in rows if r['plies_after_first_pass'] is not None)})
    result={'schema':'ahr_k18_loss_diagnosis_v1','source_report':str(path),'source_sha256':go2_file_sha(path),
        'details':details,'summary':summary,'diagnostic_positions':positions,'training_rows':0,'purpose':'consumed_development',
        'interpretation':'Post-pass captures are a pattern, not proof that passing caused a preventable loss.'}
    target=project/'reports/ahr_v2/k18_diagnosis/report.json';go2_atomic_json(target,result);return result

def go2_k18_scout_search_audit(project,model,positions,teacher=None,max_positions=8,simulations=128,device='cuda:0'):
    results=[]
    for item in positions[:int(max_positions)]:
        state=go2_state_from_dict(item['state']);evaluation=go2_eval_batch(model,[state])[0]
        scout=int(np.argmax(evaluation['prior']))
        search=go2_search(model,state,Go2SearchConfig(simulations=int(simulations),batch_size=8,wall_time_seconds=None))
        choice=int(search['action']);row={k:item[k] for k in ['game_index','profile','reason','chosen_action']}
        row.update({'scout_action':scout,'search_action':choice,'simulations':search['tree']['completed_simulations'],
            'worker_value':float(evaluation['value']),'scout_search_agree':scout==choice,'re_evaluated':True,'purpose':'consumed_development'})
        if teacher is not None:
            best=teacher.query(state);row['teacher_best_action']=best['best_action'];row['teacher_value']=best['value']
            for name,action in [('scout',scout),('search',choice),('played',int(item['chosen_action']))]:
                forced=teacher.query(state,allowed_actions=[action]);row[name+'_estimated_winrate_regret']=max(0.,(best['value']-forced['value'])/2)
        results.append(row)
    return results


def go2_k18_diagnose(project,model=None,teacher=None,max_positions=8,simulations=128,report_path=None):
    report=go2_k18_diagnosis(project,report_path)
    if model is not None:
        report['scout_search_audit']=go2_k18_scout_search_audit(project,model,report['diagnostic_positions'],teacher,
            max_positions=max_positions,simulations=simulations)
        go2_atomic_json(Path(project)/'reports/ahr_v2/k18_diagnosis/report.json',report)
    return report

def go2_k18_candidate_gate(project,run_prefix='ahr-v2-benchmark-k18-candidate-gate',games_per_candidate=4,device='cuda:0'):
    project=Path(project)
    # A bounded development gate is exposed for notebook20's detached pipeline;
    # it is not run automatically by notebook18's short validation.
    cfg=go2_bench_default_config()
    cfg.update({'run_id':run_prefix+'-plan','profiles':[{'id':'katago-v1','kind':'katago','visits':1,'games':int(games_per_candidate)}],
        'device':device,'speed_positions':3})
    cfg=go2_bench_frozen_config(project,cfg)
    champion,metadata=go2_load_selected_teacher_player(project,device=device)
    plan=go2_bench_plan(project,cfg,champion,metadata)
    outputs={}
    manifest=go2_teacher_candidate_manifest(project,metadata['run_id'])
    # Resolve by verified optimizer step, never by an unverified arbitrary path.
    for step in [8016,11724]:
        entries=[e for e in manifest['entries'] if int(e.get('optimizer_step',e.get('global_step',-1)))==step]
        if len(entries)!=1:raise RuntimeError('Required frozen candidate step missing/ambiguous: '+str(step))
        model,meta=go2_load_selected_teacher_player(project,run_id=metadata['run_id'],candidate_file=entries[0]['file'],device=device)
        config=copy.deepcopy(cfg);config['run_id']=run_prefix+'-step'+str(step)
        outputs[str(step)]=go2_bench_arena(project,config['run_id'],config,plan,model,meta,
            lambda profile,seed,board_size=9,komi=7.5:go2_bench_opponent_factory(project,profile,seed,board_size,komi,config_directory=project/'reports/ahr_v2/k18_gate_configs'))
    return outputs


## Validate target routing and interruption recovery

The fixture prepares disposable small banks from real train/dev games, compares uninterrupted versus interrupted-and-resumed array bytes, checks policy normalization and source masks, and queries the pinned teacher on a short position. It does not create the two-million-row production bank or train a model.

In [ ]:
def go2_k18_array_digests(dataset):
    result={};root=Path(dataset['cache_path'])
    for split in ['train','dev']:
        for field in GO2_K18_FIELDS:
            digest=hashlib.sha256()
            for shard in dataset[split]['shards']:
                value=np.load(root/shard['files'][field]['path'],mmap_mode='r',allow_pickle=False)
                digest.update(value.tobytes())
            result[split+'/'+field]=digest.hexdigest()
    return result

def go2_k18_validate(project,query_teacher=True):
    project=Path(project);stamp=uuid.uuid4().hex[:10]
    config={'train_positions':128,'dev_positions':64,'max_positions_per_game':8,'games_per_commit':2,
        'opponent_weights':[1]*8,'seed':181818}
    direct_config=config|{'cache_id':'k18-fixture-direct-'+stamp}
    resume_config=config|{'cache_id':'k18-fixture-resume-'+stamp}
    direct=go2_k18_prepare(project,direct_config)
    partial=go2_k18_prepare(project,resume_config,max_games=3)
    if partial['status']!='PREPARING':raise AssertionError('Fixture did not interrupt a live producer.')
    resumed=go2_k18_prepare(project,resume_config)
    if go2_k18_array_digests(direct)!=go2_k18_array_digests(resumed):raise AssertionError('Resumed rows differ from uninterrupted rows.')
    batch=go2_k18_sample_batch(resumed,np.arange(resumed['train']['count']),device='cpu')
    if not batch['policy_searched'].any() or (batch['policy_searched']&~batch['teacher_turn']).any():raise AssertionError('Searched routing fixture not exercised.')
    checkpoint=project/'checkpoints'/('ahr-v2-k18-data-'+resumed['manifest']['config']['cache_id'])
    manifest,states=go2_manifest_states(checkpoint)
    if len(states)<2 or states[manifest['latest']]['sampler']['status']!='COMPLETE':raise AssertionError('Full producer checkpoints not retained.')
    teacher_result=None
    if query_teacher:
        state=go2_initial_state(9,7.5)
        for action in [39,41,22,58]:state=go2_step(state,action)
        with Go2K18Teacher(project,visits=4,timeout_seconds=120,seed=181818) as teacher:
            first=teacher.query(state);second=teacher.query(state,allowed_actions=[first['best_action']])
        if not np.allclose(first['policy'].sum(),1,atol=1e-6) or first['policy'][~np.asarray(go2_legal_mask(state),dtype=bool)].any():raise AssertionError('Teacher policy legal normalization failed.')
        teacher_result={'visits':first['visits'],'best_action':first['best_action'],'value':first['value'],
            'forced_action':second['best_action'],'engine_model_sha256':first['identity']['assets']['model_sha256']}
    receipt={'schema':'ahr_k18_validation_v1','status':'PASS','sources':go2_k18_sources(project),
        'uninterrupted_resume_rows_identical':True,'train_rows':resumed['train']['count'],'dev_rows':resumed['dev']['count'],
        'searched_rows':int(batch['policy_searched'].sum()),'teacher_turn_rows':int(batch['teacher_turn'].sum()),
        'full_latest_two_and_best_valid':True,'test_consumed':False,'production_started':False,
        'fixture_cache_id':resumed['manifest']['config']['cache_id'],'teacher_query':teacher_result,'created_utc':go2_now()}
    go2_atomic_json(project/'reports/ahr_v2/k18_validation/receipt.json',receipt);return receipt


In [ ]:
K18_CONFIG=go2_k18_config()
DIAGNOSIS=go2_k18_diagnosis(PROJECT)
import pandas as pd
from IPython.display import display
display(pd.DataFrame(DIAGNOSIS['summary']))
VALIDATION=go2_k18_validate(PROJECT,query_teacher=True)
display(pd.DataFrame([{'Check':'Exact producer resume','Result':'PASS'},
    {'Check':'Searched teacher targets and legal support','Result':'PASS'},
    {'Check':'Latest two and best checkpoints','Result':'PASS'},
    {'Check':'Pinned teacher analysis query','Result':'PASS'},
    {'Check':'Official test consumed','Result':False}]))


## Inspect the pass/capture pattern

The bars below count observed events in the completed development matches. They do not imply that every captured group could have been saved. Loss margins use our exact completed-game score.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize':(10,3.8),'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
summary=pd.DataFrame(DIAGNOSIS['summary']).set_index('Opponent')
fig,axes=plt.subplots(1,2,figsize=(12,4))
summary[['Zero AHR stones','Large capture after first pass']].plot.bar(ax=axes[0],color=['#CC6B49','#2B6C9E'])
axes[0].set_title('Captured groups and final wipeouts',pad=45);axes[0].set_ylabel('Games');axes[0].set_xlabel('Opponent / visit cap')
axes[0].tick_params(axis='x',rotation=25);axes[0].legend(fontsize=9,loc='lower left',bbox_to_anchor=(0,1.01),frameon=False)
summary['Median score margin'].plot.bar(ax=axes[1],color='#2B6C9E')
axes[1].axhline(0,color='black',linewidth=.8);axes[1].set_ylabel('Points for AHR');axes[1].set_xlabel('Opponent / visit cap');axes[1].set_title('Median completed-game score margin');axes[1].tick_params(axis='x',rotation=25)
fig.suptitle('Consumed notebook 17 development matches: 40 GNU games and 20 per KataGo cap',fontsize=12)
fig.tight_layout();display(fig);plt.close(fig)


## Compare the planned mixture with the existing bank

The grey bars use the immutable existing bank's actual accepted counts. The blue bars are the **planned quota**, not a measured outcome or a completed new dataset. Competitive and cleanup weighting applies within each stratum as well.

In [ ]:
old_manifest=json.loads((PROJECT/'data/ahr_v2_teacher/prepared/go9x9-teacher-v1/manifest.json').read_text())
old_counts=np.asarray(old_manifest['opponent_counts']['train'],dtype=float)
mixture=pd.DataFrame({'Existing 500k bank':100*old_counts/old_counts.sum(),
    'Planned 2m bank':100*np.asarray(K18_CONFIG['opponent_weights'])},index=np.arange(8))
fig,ax=plt.subplots(figsize=(10,4))
mixture.plot.bar(ax=ax,color=['#8F969D','#2B6C9E'])
ax.set_xlabel('Source opponent stratum (higher index = later checkpoint)');ax.set_ylabel('Share of accepted training positions (%)')
ax.set_title('Existing accepted mixture versus planned production quotas')
ax.tick_params(axis='x',rotation=0);ax.legend(frameon=False)
fig.tight_layout();display(fig);plt.close(fig)


## Prepare production explicitly or continue in notebook 20

`PREPARE_PRODUCTION=False` keeps Run All to bounded diagnostics and fixtures. Notebook 20 runs preparation, controlled challenger training and fresh promotion checks in detached tmux. The production identity binds every source definition and recipe field. Do not change the recipe after a producer starts; use a new cache ID for a deliberately different experiment.

The selected player remains available in notebook 16. Notebook 19 owns challenger training; this notebook creates no model optimizer steps.

In [ ]:
if PREPARE_PRODUCTION:
    PRODUCTION_DATASET=go2_k18_prepare(PROJECT,K18_CONFIG)
    display(pd.DataFrame(go2_teacher_cache_summary(PRODUCTION_DATASET)))
else:
    display(pd.DataFrame([{'Cache':K18_CONFIG['cache_id'],'Planned train rows':K18_CONFIG['train_positions'],
        'Planned dev rows':K18_CONFIG['dev_positions'],'Strong-stratum share':sum(K18_CONFIG['opponent_weights'][4:]),
        'Production preparation':'Not started by this Run All'}]))
